# 以 ollama 執行 llama3

In [1]:
# %pip install ollama

In [37]:
import ollama
from ollama import generate
import os
import textwrap
from IPython.display import display
from IPython.display import Markdown
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np

In [38]:
model = 'llama3'
temperature = 1
top_p = 0.9
seed = 0
system_prompt = '''你是一個厲害的股市分析師，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關'''

In [39]:
#，會用繁體中文回答下列問題，當正面時回答#非常好或#好，負面時回答#不好或#非常不好，其他則是#無關
def generate_response(user_message):
  response = generate(
    model=model,
    
    prompt=f'''<|begin_of_text|><|start_header_id|>system<|end_header_id|> 

    {system_prompt}<|eot_id|> 
    <|start_header_id|>user<|end_header_id|>

    {user_message}<|eot_id|>
    <|start_header_id|>assistant<|end_header_id|>''',

    options={
      'num_predict': 128,
      'temperature': temperature,
      'top_p': top_p,
      'stop': ['<|eot_id|>'],
      'seed' : seed,
    },
  )
  # print(f'input : \n{user_message}')
  print(f'output : \n{response["response"]}')
  return response['response']

In [41]:
def lm_response(instruction):
    text = ''
    try:
        text = generate_response(instruction)
        # print(text)
    except Exception as e:
        print(e)
        pass
    # print(text)

    if text.find('非常不好') > 0:
        score = -2
    elif text.find('非常好') > 0:
        score = 2
    elif text.find('不好') > 0:
        score = -1
    elif text.find('好') > 0:
        score = 1
    else:
        score = 0

    return score  # 分數

In [36]:
# content = '''在半導體供應鏈中，IC測試業雖屬後段服務，卻是不可或缺的一環，隨著全球IDM大廠將測試產能委外的發展趨勢逐漸明顯，測試業技術挑戰性越來越高，發展空間也加大。  1顆IC的形成，包括設計、生產製造、封裝到測試，在半導體供應鏈分工過程中，測試是最後階段的工程服務，但在結合前段設計、生產製造及後段封裝的一貫化服務策略下，測試業逐漸發展成為具競爭力的分工夥伴。研發工程師需求增多研發工程師需求增多測試業做些什麼呢？1位測試廠處長表示，測試主要在檢測IC的功能，包括模擬老化、依良率差別分等級、建立有效資料供分析使用等。因IC製程微縮速度愈加快速，整合及複雜度漸高，測試業除了生產管理、設備、測試、品保、軟硬體等各類別工程師之外，各公司為了提高競爭力，漸漸投入技術研發，對研發工程師需求也增多。請繼續往下閱讀...台灣測試廠至少30家台灣測試廠至少30家台灣測試業至少超過30家，知名上市上櫃公司包括京元電、力成、泰林、矽格、欣銓、久元等，另有在海外掛牌的南茂、聯測等公司。各家公司測試產品別互有差異，有的以記憶體為主，代表性公司如力成、泰林等，京元電的邏輯產品比重較高，欣銓以12吋晶圓測試為主，日月光、矽品則以封裝為主。  業者表示，國際全球IDM（整合元件）大廠為了降低營運成本，近幾年開始，陸續釋出後段封測訂單委外代工，這使得台灣測試業營收明顯上揚，不過，隨著技術提高，測試機每台達數千萬元或上億元，測試業已成為資本密集化，同時也需透過與國際大廠策略聯盟，才有競爭力可言，這顯示測試業將走向「大者恆大」局面。應試基本條件︰電學、程式、英文應試基本條件︰電學、程式、英文起薪行情3萬2到3萬6起薪行情3萬2到3萬6記者洪友芳／專題報導工程師為測試業核心專業人力，需求人才以電子、電機、材料等理工相關科系畢業者為主，大學畢業無經驗的新人，起薪行情約3萬2千元到3萬6千元之間，依科系別及個人潛力而定；研究所畢業起薪標準約在3萬9千元上下。1位測試大廠主管表示，測試業工程師至少需要具備基本電學、軟體程式及英文能力，因設備機台是測試業最大的資本支出，也都以英文顯現，客戶有不少比例為國外廠商，工程師需懂電學、程式及英文，才能勝任測試業工作。此外，這位主管也說，測試業沒有自有產品，屬於專業服務業，以測試客戶產品居多，工程師經常需要跟客戶接洽、溝通，滿足客戶需求，工程師的服務熱忱、待人處事態度也要能屈能伸，公司的在職進修於安排專業課程之外，也會有人際溝通等強化客戶服務課程。下班時間常不固定下班時間常不固定京元電人力資源處處長柯芳蓉指出，測試業所需的專業人力，部分是跟前段的IC設計、生產製造相互競爭，基本上，前段的待遇稍高，工作性質也有別，但有的求職者偏好後段測試業，認為發展空間較大。柯芳蓉表示，測試業的工程師以常態班為主，上班時間通常是上午8點多到5點半或6點，但由於半導體業都屬責任制，測試業工程師的下班時間其實並不固定，有時因應客戶交貨趕時間，延到8、9點或半夜下班也是常有的事。論工作甘苦，辛苦是半導體業的共通處，但為客戶解決問題、令客戶滿意時，卻讓人很有工作成就感。封測類股昨於盤中紛跌停，評價： '''
# lm_response(content)

In [42]:
def calculate_return(signal, threshold):
    # signal = [日期, 評分, 價格]
    hold = 0  # 持有的股數
    balance = 10000000
    init_balance = balance

    for item in signal:
        score = float(item[1])
        price = float(item[2])

        if hold == 0:
            if score == 2:
                hold += int(balance/price)
                balance = 0
            elif score == 1:
                hold += int((balance/2)/price)
                balance = int(balance/2)
        elif hold > 0:
            if score < 0:
                balance += price*hold
                hold = 0
            elif score == 2:
                hold += int(balance/price)
                balance = 0
    
    balance += hold*price
    last_price = float(signal[len(signal) - 1][2])
    bnh = last_price / float(signal[0][2])
    r = balance/init_balance

    return round(r, 5), round(bnh, 5)

In [55]:
def write_output(content, raw_data, folder_path):
    # 輸出結果
    count = 1
    while True:
        if os.path.exists(folder_path + str(count)):
            count += 1
            continue
        else:
            os.makedirs(folder_path + str(count))
            with open(folder_path + str(count) + "/detail.txt" , "w", encoding="UTF-8") as f:
                f.write(content)
            
            for i in range(len(raw_data)):
                #寫入
                file = folder_path + str(count) + "/" + str(i+1) + ".csv"
                pd.DataFrame(raw_data[i]).to_csv(file)
                # with open(folder_path + str(count) + "/" + str(i+1), "w", encoding="UTF-8") as f:
                #     f.write(pd.DataFrame(raw_data[i]))
            print("寫入" + folder_path + str(count))
            break

'content = (f\'開始日期 {start_time.strftime("%Y%m%d")},\'\n                           f\'結束日期 {end_time.strftime("%Y%m%d")}\n\')\n                for j in range(len(stock_ids)):\n                    content += (f\'股票代號：{stock_ids[j]}\t\'\n                                f\'依照分析報酬率:{rtns[j]}\t\'\n                                f\'總額投入報酬率:{bnhs[j]}\n\')\n                content += f\'變異{i + 1}次，平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{1.170022}\n)\'\n                content += f\'原始數據{raw_data}'

In [45]:
# main cell
%time

regenerate_count = 1
prompt_mutate_count = 1

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

# stock_ids = ["1101", "2211", "2385", "2542", "2880", "2912", "3023", "3264", "5269", "8027"]
# stock_names = ["台泥", "長榮鋼", "群光", "興富發", "華南金", "統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
# stock_ids = ["2912", "3023", "3264", "5269", "8027"]
# stock_names = ["統一超", "信邦", "欣銓", "祥碩", "鈦昇"]
stock_ids = ["8027"] # 執行時間：4 min
stock_names = ["鈦昇"]

p_mutate = '業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢'
rtn_ = 0

for i in range(prompt_mutate_count):
    # 變異
    # p_mutate_2 = "業務創新計劃進行詳盡的審查，專注於預計對目標市場的影響，並探索投資者樂觀的長期增長展望刺激的股票價格上漲趨勢"

    rtns = []
    bnhs = []
    raw_data = []
    for idx, stock_id in enumerate(stock_ids):
        print(f'變異{i + 1}次\t執行{stock_id} {stock_names[idx]}')
        general_prompt = f'請依據"{p_mutate}"對{stock_names[idx]}的影響，當正面時回答#非常好或#好 ，負面時回答#不好或#非常不好，其他則是#無關'
        folder_path = "stock_news/" + stock_id + "/" + stock_id + "_news/"
        files = os.listdir(folder_path)
        sorted_files = sorted(files)

        # 載入買進價格
        df = pd.read_csv("price_history/" + stock_id + ".csv", dtype=str)
        price_np = df.to_numpy()
        prices = price_np.tolist()  # [時間,價格]

        signal_2 = []

        for price in prices:
            # 檢查日期是否在參考日期之後
            try:
                date_obj = datetime.strptime(price[0], '%Y%m%d')
            except:
                continue

            if not (start_time <= date_obj <= end_time):
                continue
            
            sig = 0
            for index, file_name in enumerate(sorted_files):
                if file_name == price[0]:
                    with open(folder_path + file_name, encoding='utf-8') as f:
                        news = f.read()
                        # print(news)
                        # prompt = general_prompt + "\n" + news
                        # gemini的回應，回傳帶有買入訊號的list
                        sig = lm_response(news)
                        print(sig)
                    break
            
            # print(sig)
            # if sig != 0 :
            signal_2.append([price[0], sig, price[1]])

        signal_2_np = np.array(signal_2)
        # 回傳 IRR 以及總額投入
        rtn, bnh = calculate_return(signal_2_np, 0)
        rtns.append(rtn)
        bnhs.append(bnh)
        raw_data.append(signal_2_np.tolist())
        # print("rtn", irr)
        # print("總額投入報酬率", bnh)

    # 輸出結果
    content = (f'開始日期 {start_time.strftime("%Y%m%d")},結束日期 {end_time.strftime("%Y%m%d")}\n')
    for j in range(len(stock_ids)):
        content += (f'股票代號：{stock_ids[j]}\t'
                    f'依照分析報酬率:{rtns[j]}\t'
                    f'總額投入報酬率:{bnhs[j]}\n')
    content += f'平均報酬率:{sum(rtns) / len(rtns)}，平均總額投入報酬率:{sum(bnhs) / len(bnhs)}\n'
    content += f'原始數據{raw_data}'
    content += f'模型資訊：{model}，溫度：{temperature}，top_p：{top_p}，seed：{seed}\n系統提示：{system_prompt}'

    write_folder = "eval_llama/output"
    write_output(content=content, raw_data=raw_data, folder_path=write_folder)

    # 判斷是否更改變異
    # if sum(rtns) / len(rtns) > rtn_:
    #     print("更改變異提示為:", p_mutate_2)
    #     p_mutate = p_mutate_2
    #     rtn_ = sum(rtns) / len(rtns)

# todo testing

CPU times: user 0 ns, sys: 0 ns, total: 0 ns
Wall time: 1.91 µs
變異1次	執行8027 鈦昇
output : 
#好
1
output : 
According to the data, 鈦昇 (8027-TW)'s December 2022 revenue is #好.
1
output : 
#好！根據鈦昇(8027-TW)的2022年12月營收資料，單位為千元，我可以看到是一個健康的數字。
1
output : 
**#好**

Based on the 2023 February revenue data of鈦昇 (8027-TW), I'm pleased to see a positive trend! With a strong performance in the first two months of the year, I'd say it's a good sign for investors. Would you like me to analyze the numbers further?
1
output : 
I see that we are discussing the financial report and dividend distribution of 鈦昇 (8027) semiconductor equipment manufacturer. The key points mentioned include:

1. The company's board of directors has approved the 2021 financial report and dividend distribution.
2. The company's revenue and net income have both reached new highs, with each share after-tax profit being NT$3.94.
3. The board of directors has decided to distribute a cash dividend of NT$2 per share, which is higher than

TypeError: write() argument must be str, not DataFrame